### MLP 

In [1]:
import torch 
from fastai.vision.all import *

/home/cgb4/anaconda3/envs/py38r40/lib/python3.8/site-packages/torch/cuda/__init__.py:106: UserWarning: 
NVIDIA GeForce RTX 3090 with CUDA capability sm_86 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70.
If you want to use the NVIDIA GeForce RTX 3090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


In [ ]:
path = untar_data(URLs.MNIST_SAMPLE)

In [ ]:
path.ls()

`-` list

In [ ]:
threes=(path/'train'/'3').ls()
sevens=(path/'train'/'7').ls()

`-` list $\to$ image 

In [ ]:
Image.open(threes[0])

`-` image $\to$ tensor

In [ ]:
tensor(Image.open(threes[0]))

- tensor는 fastai에서 구현한 함수. 리턴값은 파이토치에서 우리가 쓰던 그 텐서

`-` 여러개를 처리하여보자. 

In [ ]:
seven_tensor = torch.stack([tensor(Image.open(i)) for i in sevens]).float()/255
three_tensor = torch.stack([tensor(Image.open(i)) for i in threes]).float()/255

`-` $X$와 $y$를 만들자.

In [ ]:
seven_tensor.shape, three_tensor.shape

In [ ]:
y=torch.tensor([0]*6265 + [1]*6131 )

In [ ]:
X=torch.vstack([seven_tensor,three_tensor])

In [ ]:
X=X.reshape(12396,28*28)

In [ ]:
X.shape

In [ ]:
y.shape

`-` 모형

${\bf X} \to \Big\{ {\bf WX}+{\bf b} \to f({\bf WX} +{\bf b}) \Big\}  \to  \dots \to {\bf y}$

아키텍처: 적당히 깊게.. + 적당히 넓게.. + 표현력이 충분하면서도.. + 과적합은 하지 않도록 (저도 잘 모르겠어요) 

손실함수: BCEloss

옵티마이저: ADAM

`-` 교재의 모형

In [ ]:
import graphviz

In [ ]:
def gv(s): return graphviz.Source('digraph G{ rankdir="LR"' + s + '; }')

In [ ]:
#collapse
gv('''
splines=line
subgraph cluster_1{
    style=filled;
    color=lightgrey;
    "x1"
    "x2"
    ".."
    "x784"
    label = "Layer 0"
}
subgraph cluster_2{
    style=filled;
    color=lightgrey;
    "x1" -> "node1"
    "x2" -> "node1"
    ".." -> "node1"
    
    "x784" -> "node1"
    "x1" -> "node2"
    "x2" -> "node2"
    ".." -> "node2"
    "x784" -> "node2"
    
    "x1" -> "..."
    "x2" -> "..."
    ".." -> "..."
    "x784" -> "..."

    "x1" -> "node30"
    "x2" -> "node30"
    ".." -> "node30"
    "x784" -> "node30"


    label = "Layer 1: ReLU"
}
subgraph cluster_3{
    style=filled;
    color=lightgrey;
    "node1" -> "y"
    "node2" -> "y"
    "..." -> "y"
    "node30" -> "y"
    label = "Layer 2: Sigmoid"
}
''')

In [ ]:
net = torch.nn.Sequential(
    torch.nn.Linear(28*28,30),
    torch.nn.ReLU(),
    torch.nn.Linear(30,1), 
    torch.nn.Sigmoid()
)

In [ ]:
X.to("cuda:0")

In [ ]:
optimizer=torch.optim.Adam(net.parameters())

In [ ]:
losses=[]
for epoc in range(20): 
    ## 1 
    yhat=net(X) 
    ## 2 
    loss= -torch.mean(y*torch.log(yhat) + (1-y)*torch.log(1-yhat)) ## BCEloss
    ## 3 
    loss.backward()
    ## 4 
    optimizer.step() 
    net.zero_grad()
    ## 5
    losses.append(loss.item())

In [ ]:
plt.plot(losses)

In [ ]:
ds=torch.utils.data.TensorDataset(X,y)

In [ ]:
dl=torch.utils.data.DataLoader(ds,batch_size=64,shuffle=True)

In [ ]:
X.shape

In [ ]:
losses=[]
i=0
j=0
for epoc in range(100): 
    for xx,yy in dl:
        ## 1 
        yhat=net(xx) 
        ## 2 
        loss= -torch.mean(yy*torch.log(yhat) + (1-yy)*torch.log(1-yhat)) ## BCEloss
        ## 3 
        loss.backward()
        ## 4 
        optimizer.step() 
        net.zero_grad()
        ## 5
        losses.append(loss.item())

In [ ]:
plt.plot(y,'.')
plt.plot(net(X).data,'.')